# Domain Adaptation

In [ ]:
import sys
sys.path.append('..')
from src.dataset.dataset import (
    build_hsi_testloader,
    build_hsi_dataloader,
    HSIDataset,
    build_FIVES_random_crops_dataloaders,
)
from src.util.constants import FIVES_RANDOM_CROPS_DIR, RING_LABELS_DIR
from torch.utils.data import DataLoader

trainloader_source, validationloader_source, testloader_source = (
    build_FIVES_random_crops_dataloaders(
        batch_size=16,
        num_channels=1,
        load_from_path=FIVES_RANDOM_CROPS_DIR,
    )
)

window = (500, 600)
target_augmented= True
trainloader_target = build_hsi_dataloader(
    batch_size=8,
    train_split=1,
    val_split=0,
    test_split=0,
    window=window,
    exclude_labeled_data=True,
    augmented=target_augmented,
    ring_label_dir=RING_LABELS_DIR,
)[0]

testloader_target = build_hsi_testloader(
    batch_size=1,
    window=window,
)

print(len(trainloader_target.dataset), len(testloader_target.dataset))

## Evaluation of Model without Domain Adaptation on Target Testset

### Transfer Learning

In [ ]:
from src.util.segmentation_util import (
    evaluate_model,
    build_segmentation_model,
    load_model,
)
from src.util.constants import MODELS_DIR
import torch

device = torch.device("cuda:3" if torch.cuda.is_available() else "cpu")
model_path = (
    MODELS_DIR
    / f"Linknet-timm-regnetx_320-512x512-augmented-with-random-crops-single-channel-focal.pth"
)
model = build_segmentation_model(
    encoder="timm-regnetx_320",
    architecture="Linknet",
    device=device,
    in_channels=1,
    path=model_path,
)
model.eval()
evaluate_model(model, testloader_target, device, with_wandb=False)

### Ensemble Model

In [ ]:
from src.model.HSI_models.HSI_Net import SegmentationModelWithWindowing
from src.util.segmentation_util import build_segmentation_model, evaluate_model
from src.model.HSI_models.ensemble_model import EnsembleModelAverage
import segmentation_models_pytorch as smp
import torch
from src.util.constants import HELICOID_WITH_LABELS_DIR, MODELS_DIR

path = HELICOID_WITH_LABELS_DIR
testset = HSIDataset(path, with_gt=True)
testset.crop_dataset()
testloader_target = DataLoader(testset, batch_size=1, shuffle=False)

encoder = "timm-regnetx_320"
architecture = "Linknet"
device = "cuda:5" if torch.cuda.is_available() else "cpu"

model_path_segmenter = (
    MODELS_DIR
    / f"{architecture}-{encoder}-512x512-augmented-with-random-crops-single-channel-focal.pth"
)

segmenter_single_channel = build_segmentation_model(
    encoder, architecture, device, in_channels=1, path=model_path_segmenter
)
segmenter_single_channel.eval()

window1 = (400, 500)
window2 = (500, 600)
window3 = (600, 700)
window4 = (700, 800)
window5 = (800, 900)
window6 = (900, 1000)
window_full = (400, 1100)


model_windowing_400to500 = SegmentationModelWithWindowing(
    segmenter_single_channel, window=testset.get_window_from_wavelengths(window1)
)
model_with_windowing_500to600 = SegmentationModelWithWindowing(
    segmenter_single_channel, window=testset.get_window_from_wavelengths(window2)
)
model_windowing_600to700 = SegmentationModelWithWindowing(
    segmenter_single_channel, window=testset.get_window_from_wavelengths(window3)
)
model_windowing_700to800 = SegmentationModelWithWindowing(
    segmenter_single_channel, window=testset.get_window_from_wavelengths(window4)
)
model_windowing_800to900 = SegmentationModelWithWindowing(
    segmenter_single_channel, window=testset.get_window_from_wavelengths(window5)
)
model_windowing_900to1000 = SegmentationModelWithWindowing(
    segmenter_single_channel, window=testset.get_window_from_wavelengths(window6)
)
model_window_full = SegmentationModelWithWindowing(
    segmenter_single_channel, window=testset.get_window_from_wavelengths(window_full)
)

model = EnsembleModelAverage(
    [
        model_windowing_400to500,
        model_with_windowing_500to600,
        model_windowing_600to700,
        model_windowing_700to800,
        model_windowing_900to1000,
        model_window_full,
    ]
)
evaluate_model(model, testloader_target, device, with_wandb=False)

## Domain Adaptation Approaches:
### GRL domain adaptation

In [ ]:
import matplotlib.pyplot as plt
import torch
from src.training.domain_adaptation_training import model_pipeline
from src.util.constants import MODELS_DIR

encoder = "timm-regnetx_320"
architecture = "Linknet"
device = torch.device("cuda:4" if torch.cuda.is_available() else "cpu")
lambda_param = 0.5
learning_rate = 0.001
loss = "BCE"
epochs = 10
in_channels = 1
hidden_dim = 256
pretrained_path = (
    MODELS_DIR
    / f"Linknet-timm-regnetx_320-512x512-augmented-with-random-crops-single-channel-focal.pth"
)

config = {
    "model": "GRL-{architecture}-{encoder}-window_{window[0]}-{window[1]}-pretrained-random_crops-bloodvessel_ratio01".format(
        architecture=architecture, encoder=encoder, window=window
    ),
    "encoder": encoder,
    "architecture": architecture,
    "learning_rate": learning_rate,
    "optimizer": "Adam",
    "lambda_param": lambda_param,
    "batch_size_source": trainloader_source.batch_size,
    "batch_size_target": trainloader_target.batch_size,
    "hidden_dim": hidden_dim,
    "epochs": epochs,
    "loss": loss,
    "in_channels": in_channels,
    "pretrained": pretrained_path,
}

model, train_losses, domain_losses, val_losses_source, val_losses_target = (
    model_pipeline(
        trainloader_source,
        validationloader_source,
        testloader_source,
        trainloader_target,
        testloader_target,
        config,
        project="domain-adaptation",
        device=device,
        batch_print=10,
        evaluate=True,
        with_overlays=True,
    )
)

# Plotting the training and validation loss curves
plt.figure(figsize=(10, 6))
plt.plot(train_losses, label="Combined Training Loss")
plt.plot(domain_losses, label="Domain Loss")
plt.plot(val_losses_source, label="Validation Loss Source")
plt.plot(val_losses_target, label="Validation Loss Target")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Training and Validation Loss Curves")
plt.legend()
plt.show()

### FADA:

In [ ]:
import matplotlib.pyplot as plt
from src.training.train_FADA_unsupervised import model_pipeline
from src.util.constants import MODELS_DIR, FIVES_RANDOM_CROPS_DIR
import torch

encoder = "timm-regnetx_320"
architecture = "Linknet"
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
pretrained_path = (
    MODELS_DIR
    / f"Linknet-timm-regnetx_320-512x512-augmented-with-random-crops-single-channel-focal.pth"
)
dataset_path = FIVES_RANDOM_CROPS_DIR

config = {
    "model": "FADA-With-FeatureConvReducer-1x1Conv",
    "encoder": encoder,
    "architecture": architecture,
    "in_channels": 1,
    "optimizer": "Adam",
    "batch_size_source": 16,
    "batch_size_target": 8,
    "learning_rate_fea": 0.001,
    "learning_rate_cls": 0.001,
    "learning_rate_dis": 0.0001,
    "ndf": 512,
    "epochs": 1,
    "seg_loss": "BCE",
    "pretrained": pretrained_path,
    "augmented": True,
    "dataset_path": dataset_path,
    "feature_conv_reducer": "1x1conv",
    "cycle_loss_hyperparams": {
        "lr_F": 0.001,
        "lr_G": 0.001,
        "lambda_F": 0.5,
        "lambda_G": 0.5,
    },
}


model, train_losses, domain_losses, val_losses_source, val_losses_target = (
    model_pipeline(
        config,
        project="domain-adaptation",
        device=device,
        batch_print=5,
        evaluate=True,
        with_overlays=True,
        save_wandb=False,
        save_path="models/FeatureConvReducer",
    )
)

plt.figure(figsize=(10, 6))
plt.plot(train_losses, label="Combined Training Loss")
plt.plot(domain_losses, label="Domain Loss")
plt.plot(val_losses_source, label="Validation Loss Source")
plt.plot(val_losses_target, label="Validation Loss Target")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Training and Validation Loss Curves")
plt.legend()
plt.show()

### Supervised Domain Adapatation

#### GRL

In [ ]:
from torch.utils.data import DataLoader, Subset
import torch
from src.training.supervised_domain_adaptation import model_pipeline

trainloader_target_labeled = DataLoader(
    Subset(testset, [0]),
    batch_size=1,
    shuffle=True,
)
testloader_target = DataLoader(
    Subset(testset, [1, 2, 3, 4]), batch_size=1, shuffle=False
)

encoder = "timm-regnetx_320"
architecture = "Linknet"
device = torch.device("cuda:4" if torch.cuda.is_available() else "cpu")
lambda_param = 0.5
learning_rate = 0.001
loss = "Dice"
epochs = 10
in_channels = 1
hidden_dim = 256

config = {
    "model": "GRL-{architecture}-{encoder}-window_{window[0]}-{window[1]}-pretrained-random_crops-bloodvessel_ratio01-supervised".format(
        architecture=architecture, encoder=encoder, window=window
    ),
    "encoder": encoder,
    "architecture": architecture,
    "learning_rate": learning_rate,
    "optimizer": "Adam",
    "lambda_param": lambda_param,
    "batch_size_source": trainloader_source.batch_size,
    "batch_size_target": trainloader_target.batch_size,
    "hidden_dim": hidden_dim,
    "epochs": epochs,
    "loss": loss,
    "in_channels": in_channels,
    "pretrained": './models/Linknet-timm-regnetx_320-512x512-augmented-with-random-crops-single-channel-focal.pth',
    "supervised": True,
}

model, train_losses, domain_losses, val_losses_source, val_losses_target = model_pipeline(
    trainloader_source,
    validationloader_source,
    testloader_source,
    trainloader_target_labeled,
    trainloader_target,
    testloader_target,
    config,
    project="supervised-domain-adaptation",
    device=device,
    batch_print=1,
    evaluate=True,
    with_overlays=True,
)

#### FADA

In [ ]:
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader, Subset
from src.training.train_FADA_supervised import model_pipeline

train_indices = [0]
trainloader_target_labeled = DataLoader(
    Subset(testset, train_indices),
    batch_size=1,
    shuffle=True,
)
testloader_target = DataLoader(
    Subset(testset, list({0, 1, 2, 3, 4} - set(train_indices))),
    batch_size=1,
    shuffle=False,
)

encoder = "timm-regnetx_320"
architecture = "Linknet"
device = torch.device("cuda:4" if torch.cuda.is_available() else "cpu")
epochs = 10
loss = "BCE"
in_channels = 1
ndf = 512
learning_rate_fea = 0.001
learning_rate_cls = 0.001
learning_rate_dis = 0.0001

config = {
    "model": "FADA-{architecture}-{encoder}-window_{window[0]}-{window[1]}_pretrained-augmented_target-random_crops_bloodvessel_ratio01".format(
        architecture=architecture, encoder=encoder, window=window
    ),
    "encoder": encoder,
    "architecture": architecture,
    "in_channels": in_channels,
    "optimizer": "Adam",
    "batch_size_source": trainloader_source.batch_size,
    "batch_size_target": trainloader_target.batch_size,
    "learning_rate_fea": learning_rate_fea,
    "learning_rate_cls": learning_rate_cls,
    "learning_rate_dis": learning_rate_dis,
    "ndf": ndf,
    "epochs": epochs,
    "seg_loss": loss,
    "pretrained": "./models/Linknet-timm-regnetx_320-512x512-augmented-with-random-crops-single-channel-focal.pth",
    "target_augmented": target_augmented,
    "train_indices": train_indices,
}


model, train_losses, domain_losses, val_losses_source, val_losses_target = (
    model_pipeline(
        trainloader_source,
        validationloader_source,
        testloader_source,
        trainloader_target_labeled,
        trainloader_target,
        testloader_target,
        config,
        project="supervised-domain-adaptation",
        device=device,
        batch_print=10,
        evaluate=True,
        with_overlays=True,
    )
)

plt.figure(figsize=(10, 6))
plt.plot(train_losses, label="Combined Training Loss")
plt.plot(domain_losses, label="Domain Loss")
plt.plot(val_losses_source, label="Validation Loss Source")
plt.plot(val_losses_target, label="Validation Loss Target")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Training and Validation Loss Curves")
plt.legend()
plt.show()

### Gaussian Autoencoder

#### Training

In [ ]:
from dataset import build_hsi_dataloader, HSIDataset, build_hsi_testloader
from src.training.train_autoencoder import model_pipeline_autoencoder
import torch

device = torch.device("cuda:2" if torch.cuda.is_available() else "cpu")
config = {
    "model": "gcr-comic-sweep-trained-longer",
    "optimizer": "Adam",
    "lr_encoder": 0.003991856255637263,
    "lr_decoder": 0.0001405199848490131,
    "batch_size": 4,
    "epochs": 25,
    "mu": [335, 267, 131],
    "sigma": [1, 1, 1],
}

model_pipeline_autoencoder(
    config,
    project="hsi-autoencoder",
    epochs=25,
    device=device,
    batch_print=5,
    save_wandb=False
)

#### Gaussian reduction + DA

In [ ]:
import matplotlib.pyplot as plt
from src.training.train_FADA_unsupervised import model_pipeline
import torch
from src.dataset.dataset import build_hsi_dataloader, build_hsi_testloader, build_FIVES_random_crops_dataloaders

trainloader_source, validationloader_source, testloader_source = (
    build_FIVES_random_crops_dataloaders(
        batch_size=16,
        num_channels=3,
        load_from_path='data/FIVES_random_crops_threshold01'
    )
)

trainloader_hsi = build_hsi_dataloader(
    batch_size=8,
    train_split=1,
    val_split=0,
    test_split=0,
    exclude_labeled_data=True,
    augmented=True,
)[0]

testloader_hsi = build_hsi_testloader()

encoder = "timm-regnetx_320"
architecture = "Linknet"
device = torch.device("cuda:2" if torch.cuda.is_available() else "cpu")
epochs = 10
loss = "BCE"
in_channels = 3
ndf = 512
learning_rate_fea = 0.007053148664641283
learning_rate_cls = 0.0014884108364363155
learning_rate_dis = 0.00043925067706149473

config = {
    "model": "FADA-{architecture}-{encoder}-gcr_comic-sweep_pretrained-augmented_target-random_crops_bloodvessel_ratio01-unsupervised".format(
        architecture=architecture, encoder=encoder
    ),
    "encoder": encoder,
    "architecture": architecture,
    "in_channels": in_channels,
    "optimizer": "Adam",
    "batch_size_source": trainloader_source.batch_size,
    "batch_size_target": trainloader_hsi.batch_size,
    "learning_rate_fea": learning_rate_fea,
    "learning_rate_cls": learning_rate_cls,
    "learning_rate_dis": learning_rate_dis,
    "ndf": ndf,
    "epochs": epochs,
    "seg_loss": loss,
    "pretrained": "./models/serene-sweep-9.pth",
    "target_augmented": True,
    "gaussian": 'models/gcr_comic-sweep.pth',
    "mu_gaussian": [334.19, 266.26, 131.40],
    "sigma_gaussian": [2.92, 2.02, 2.00],
}

model, train_losses, domain_losses, val_losses_source, val_losses_target = (
    model_pipeline(
        trainloader_source,
        validationloader_source,
        testloader_source,
        trainloader_hsi,
        testloader_hsi,
        config,
        project="domain-adaptation",
        device=device,
        batch_print=5,
        evaluate=True,
        with_overlays=True,
    )
)

plt.figure(figsize=(10, 6))
plt.plot(train_losses, label="Combined Training Loss")
plt.plot(domain_losses, label="Domain Loss")
plt.plot(val_losses_source, label="Validation Loss Source")
plt.plot(val_losses_target, label="Validation Loss Target")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Training and Validation Loss Curves")
plt.legend()
plt.show()

### Conv Autoencoder

#### Training

In [ ]:
from dataset import build_hsi_dataloader, HSIDataset, build_hsi_testloader
from src.model.dimensionality_reduction.autoencoder import (
    ConvAutoEncoder,
)
from src.training.train_autoencoder import model_pipeline_autoencoder
import torch
from torch.utils.data import DataLoader

trainloader_hsi = build_hsi_dataloader(
    batch_size=8,
    train_split=1,
    val_split=0,
    test_split=0,
    exclude_labeled_data=True,
    augmented=True,
)[0]

testloader_hsi = build_hsi_testloader()

lr = 0.001
gaussian = ConvAutoEncoder()
device = torch.device("cuda:2" if torch.cuda.is_available() else "cpu")
criterion = torch.nn.MSELoss()
optimizer = torch.optim.Adam(gaussian.parameters(), lr=lr)


config = {
    "model": "convReducer",
    "optimizer": "Adam",
    "learning_rate": lr,
    "batch_size": trainloader_hsi.batch_size,
    "epochs": 10,
}

model_pipeline_autoencoder(
    gaussian,
    trainloader_hsi,
    testloader_hsi,
    criterion,
    optimizer,
    config,
    project="hsi-autoencoder",
    device=device,
    model_name="convReducer",
    batch_print=5,
)

## Evaluation

In [ ]:
from src.dataset.dataset import build_hsi_testloader
from src.util.segmentation_util import (
    build_segmentation_model,
    show_interactive_overlay,
)
from src.util.constants import RING_LABELS_DIR, MODELS_DIR
import torch

testloader_target, testloader_ring_label = build_hsi_testloader(
    window=(500, 600),
    batch_size=1,
    ring_label_dir=RING_LABELS_DIR,
)

device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")
model_path = (
    MODELS_DIR
    / "Linknet-timm-regnetx_320-512x512-augmented-with-random-crops-single-channel-focal.pth"
)
model = build_segmentation_model(
    "timm-regnetx_320", "Linknet", device, in_channels=1, path=model_path
)
# model = build_FADA_segmentation_model(
#     architecture="Linknet",
#     encoder="timm-regnetx_320",
#     device=device,
#     in_channels=3,
#     path="models/FADA-Linknet-timm-regnetx_320-gcr_comic-sweep_pretrained-augmented_target-random_crops_bloodvessel_ratio01-unsupervised.pth",
#     channel_reducer_path="models/gcr_comic-sweep.pth",
# )

index = 0
show_interactive_overlay(
    model,
    data=testloader_target.dataset[index],
    device=device,
    title="test",
    ring_data=testloader_ring_label.dataset[index][1],
)
# evaluate_model_with_postprocessing(
#     model, testloader_target, testloader_ring_label, device, with_wandb=False
# )

### Approach 1: FADA-Grayscale-Windowing

In [ ]:
from src.util.segmentation_util import cross_testing, get_model_paths
from src.model.FADA.segmentation_model import build_FADA_segmentation_model
from src.util.constants import MODELS_DIR

model_params = {
    "architecture": "Linknet",
    "encoder": "timm-regnetx_320",
    "in_channels": 1,
}
cross_testing(
    get_model_paths(f"{MODELS_DIR}/approach1"),
    window=(500, 600),
    device="cuda:1",
    build_func=build_FADA_segmentation_model,
    model_params=model_params,
)

### Approach 2: FADA-Grayscale-Windowing-NoPretraining

In [ ]:
from src.util.segmentation_util import cross_testing, get_model_paths
from src.model.FADA.segmentation_model import build_FADA_segmentation_model
from src.util.constants import MODELS_DIR

model_params = {
    "architecture": "Linknet",
    "encoder": "timm-regnetx_320",
    "in_channels": 1,
}
cross_testing(
    get_model_paths(f"{MODELS_DIR}/approach2"),
    window=(500, 600),
    device="cuda:1",
    build_func=build_FADA_segmentation_model,
    model_params=model_params,
)

### Approach 3: FADA-3ChannelWindowing

In [ ]:
from src.util.segmentation_util import cross_testing, get_model_paths
from src.model.FADA.segmentation_model import build_FADA_segmentation_model
from src.util.constants import MODELS_DIR

model_params = {
    "architecture": "Linknet",
    "encoder": "timm-regnetx_320",
    "in_channels": 3,
    "windows_in_nm": [[600, 1000], [500, 600], [400, 500]],
}
cross_testing(
    get_model_paths(f"{MODELS_DIR}/approach3"),
    build_func=build_FADA_segmentation_model,
    device="cuda:1",
    model_params=model_params,
)

### Approach 4: FADA-Grayscale-Windowing-400_500

In [ ]:
from src.util.segmentation_util import cross_testing, get_model_paths
from src.model.FADA.segmentation_model import build_FADA_segmentation_model
from src.util.constants import MODELS_DIR


model_params = {
    "architecture": "Linknet",
    "encoder": "timm-regnetx_320",
    "in_channels": 1,
}
cross_testing(
    get_model_paths(f"{MODELS_DIR}/approach4"),
    window=(400, 500),
    device="cuda:1",
    build_func=build_FADA_segmentation_model,
    model_params=model_params,
)

### Approach 5: FADA-Grayscale-Windowing-600_800

In [ ]:
from src.util.segmentation_util import cross_testing, get_model_paths
from src.model.FADA.segmentation_model import build_FADA_segmentation_model
from src.util.constants import MODELS_DIR


model_params = {
    "architecture": "Linknet",
    "encoder": "timm-regnetx_320",
    "in_channels": 1,
}
cross_testing(
    get_model_paths(f"{MODELS_DIR}/approach5"),
    window=(600, 800),
    device="cuda:1",
    build_func=build_FADA_segmentation_model,
    model_params=model_params,
)

### Approach 6: Ensemble Model

In [ ]:
from src.util.segmentation_util import cross_testing, get_model_paths, build_ensemble_model
from src.util.constants import MODELS_DIR

model_params = {}
cross_testing(
    get_model_paths(f"{MODELS_DIR}/approach6"),
    device="cuda:1",
    build_func=build_ensemble_model,
    model_params=model_params,
)


### Approach 7: FADA-Grayscale with CL Dice

In [ ]:
from src.util.segmentation_util import cross_testing, get_model_paths
from src.model.FADA.segmentation_model import build_FADA_segmentation_model
from src.util.constants import MODELS_DIR


model_params = {
    "architecture": "Linknet",
    "encoder": "timm-regnetx_320",
    "in_channels": 1,
}
cross_testing(
    get_model_paths(f"{MODELS_DIR}/approach7"),
    window=(500, 600),
    device="cuda:0",
    build_func=build_FADA_segmentation_model,
    model_params=model_params,
)

### Approach 8: FADA-Grayscale-CycleGAN-CNN

In [ ]:
from src.util.segmentation_util import cross_testing, get_model_paths
from src.model.FADA.segmentation_model import (
    build_FADA_segmentation_model_with_dimensionality_reduction,
)
from src.util.constants import MODELS_DIR

model_params = {
    "architecture": "Linknet",
    "encoder": "timm-regnetx_320",
    "in_channels": 1,
    "feature_conv_reducer": "cnn",
    "kernel_size": 3,
}

cross_testing(
    get_model_paths(f"{MODELS_DIR}/approach8"),
    device="cuda:0",
    build_func=build_FADA_segmentation_model_with_dimensionality_reduction,
    model_params=model_params,
)

### Approach 9: FADA-Grayscale-CycleGAN-1x1Conv

In [ ]:
from src.util.segmentation_util import cross_testing, get_model_paths
from src.model.FADA.segmentation_model import (
    build_FADA_segmentation_model_with_dimensionality_reduction,
)
from src.util.constants import MODELS_DIR


model_params = {
    "architecture": "Linknet",
    "encoder": "timm-regnetx_320",
    "in_channels": 1,
    "feature_conv_reducer": "1x1conv",
}

cross_testing(
    get_model_paths(f"{MODELS_DIR}/approach9"),
    device="cuda:0",
    build_func=build_FADA_segmentation_model_with_dimensionality_reduction,
    model_params=model_params,
)

### Baseline: Transfer Learning

#### Grayscale

In [ ]:
from src.util.segmentation_util import cross_testing, build_segmentation_model, get_model_paths
from src.util.constants import MODELS_DIR

model_params = {
    "architecture": "Linknet",
    "encoder": "timm-regnetx_320",
    "in_channels": 1,
}
cross_testing(
    get_model_paths(f"{MODELS_DIR}/baseline_grayscale"),
    window=(500, 600),
    device="cuda:1",
    build_func=build_segmentation_model,
    model_params=model_params,
)

#### 3 Channel Windowing

In [ ]:
from src.util.segmentation_util import cross_testing, get_model_paths
from src.model.FADA.segmentation_model import (
    build_baseline_segmentation_model_with_window_reducer,
)
from src.util.constants import MODELS_DIR


model_params = {
    "architecture": "Linknet",
    "encoder": "timm-regnetx_320",
    "in_channels": 3,
    "windows_in_nm": [[600, 1000], [500, 600], [400, 500]],
}
cross_testing(
    get_model_paths(f"{MODELS_DIR}/baseline_3ChannelWindowing"),
    build_func=build_baseline_segmentation_model_with_window_reducer,
    device="cuda:1",
    model_params=model_params,
)

### Plot Results

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
approach1 = {
    0: {"Precision": 0.5352, "Recall": 0.6670, "DiceScore": 0.5910, "Accuracy": 0.8196, "CLDiceScore": 0.5533},
    1: {"Precision": 0.5303, "Recall": 0.7830, "DiceScore": 0.6316, "Accuracy": 0.7854, "CLDiceScore": 0.5946},
    2: {"Precision": 0.5292, "Recall": 0.6412, "DiceScore": 0.5739, "Accuracy": 0.8251, "CLDiceScore": 0.5309},
    3: {"Precision": 0.5170, "Recall": 0.6194, "DiceScore": 0.5522, "Accuracy": 0.8223, "CLDiceScore": 0.5161},
    4: {"Precision": 0.5572, "Recall": 0.6122, "DiceScore": 0.5679, "Accuracy": 0.8267, "CLDiceScore": 0.5054},
}

approach2 = {
    0: {"Precision": 0.5215, "Recall": 0.3836, "DiceScore": 0.4391, "Accuracy": 0.8105, "CLDiceScore": 0.3675},
    1: {"Precision": 0.5370, "Recall": 0.6201, "DiceScore": 0.5739, "Accuracy": 0.7871, "CLDiceScore": 0.5359},
    2: {"Precision": 0.5145, "Recall": 0.4671, "DiceScore": 0.4838, "Accuracy": 0.8076, "CLDiceScore": 0.4206},
    3: {"Precision": 0.4186, "Recall": 0.6554, "DiceScore": 0.5100, "Accuracy": 0.7725, "CLDiceScore": 0.4902},
    4: {"Precision": 0.4513, "Recall": 0.5585, "DiceScore": 0.4962, "Accuracy": 0.7865, "CLDiceScore": 0.4297},
}

approach3 = {
    0: {"Precision": 0.5057, "Recall": 0.7008, "DiceScore": 0.5851, "Accuracy": 0.8076, "CLDiceScore": 0.5838},
    1: {"Precision": 0.5496, "Recall": 0.6445, "DiceScore": 0.5901, "Accuracy": 0.7969, "CLDiceScore": 0.5533},
    2: {"Precision": 0.4882, "Recall": 0.6959, "DiceScore": 0.5640, "Accuracy": 0.7975, "CLDiceScore": 0.5577},
    3: {"Precision": 0.4675, "Recall": 0.5350, "DiceScore": 0.4882, "Accuracy": 0.8040, "CLDiceScore": 0.4738},
    4: {"Precision": 0.4793, "Recall": 0.5797, "DiceScore": 0.5174, "Accuracy": 0.8007, "CLDiceScore": 0.5042},
}

approach4 = {
    0: {"Precision": 0.4900, "Recall": 0.4502, "DiceScore": 0.4587, "Accuracy": 0.8090, "CLDiceScore": 0.4030},
    1: {"Precision": 0.6636, "Recall": 0.5839, "DiceScore": 0.6185, "Accuracy": 0.8303, "CLDiceScore": 0.5100},
    2: {"Precision": 0.5159, "Recall": 0.7684, "DiceScore": 0.6044, "Accuracy": 0.8017, "CLDiceScore": 0.5479},
    3: {"Precision": 0.5130, "Recall": 0.6696, "DiceScore": 0.5690, "Accuracy": 0.8173, "CLDiceScore": 0.4701},
    4: {"Precision": 0.5580, "Recall": 0.4937, "DiceScore": 0.5058, "Accuracy": 0.8265, "CLDiceScore": 0.4351},
}

approach5 = {
    0: {"Precision": 0.6166, "Recall": 0.5479, "DiceScore": 0.5725, "Accuracy": 0.8410, "CLDiceScore": 0.4437},
    1: {"Precision": 0.6738, "Recall": 0.6101, "DiceScore": 0.6325, "Accuracy": 0.8346, "CLDiceScore": 0.4865},
    2: {"Precision": 0.5898, "Recall": 0.6921, "DiceScore": 0.6357, "Accuracy": 0.8463, "CLDiceScore": 0.5393},
    3: {"Precision": 0.5557, "Recall": 0.6573, "DiceScore": 0.6011, "Accuracy": 0.8414, "CLDiceScore": 0.4991},
    4: {"Precision": 0.6222, "Recall": 0.4784, "DiceScore": 0.5318, "Accuracy": 0.8394, "CLDiceScore": 0.3968},
}


approach6 = {
    0: {"Precision": 0.5827, "Recall": 0.3645, "DiceScore": 0.4380, "Accuracy": 0.8192, "CLDiceScore": 0.3688},
    1: {"Precision": 0.7914, "Recall": 0.4309, "DiceScore": 0.5500, "Accuracy": 0.8380, "CLDiceScore": 0.3970},
    2: {"Precision": 0.7020, "Recall": 0.3966, "DiceScore": 0.5029, "Accuracy": 0.8408, "CLDiceScore": 0.3776},
    3: {"Precision": 0.6902, "Recall": 0.4428, "DiceScore": 0.5340, "Accuracy": 0.8612, "CLDiceScore": 0.3941},
    4: {"Precision": 0.6864, "Recall": 0.3987, "DiceScore": 0.4959, "Accuracy": 0.8489, "CLDiceScore": 0.3489},
}

approach7= {
    0: {"Precision": 0.4815, "Recall": 0.6294, "DiceScore": 0.5438, "Accuracy": 0.7948, "CLDiceScore": 0.5585},
    1: {"Precision": 0.4655, "Recall": 0.7588, "DiceScore": 0.5746, "Accuracy": 0.7363, "CLDiceScore": 0.6051},
    2: {"Precision": 0.4996, "Recall": 0.6755, "DiceScore": 0.5114, "Accuracy": 0.7841, "CLDiceScore": 0.5233},
    3: {"Precision": 0.0, "Recall": 0.2905, "DiceScore": 0.3079, "Accuracy": 0.7928, "CLDiceScore": 0.0},
    4: {"Precision": 0.5480, "Recall": 0.6907, "DiceScore": 0.5480, "Accuracy": 0.7845, "CLDiceScore": 0.5571},
}

approach8 = {
    0: {"Precision": 0.6755, "Recall": 0.5216, "DiceScore": 0.5808, "Accuracy": 0.8527, "CLDiceScore": 0.4120},
    1: {"Precision": 0.6439, "Recall": 0.6085, "DiceScore": 0.6142, "Accuracy": 0.8228, "CLDiceScore": 0.4890},
    2: {"Precision": 0.5723, "Recall": 0.6578, "DiceScore": 0.5804, "Accuracy": 0.8194, "CLDiceScore": 0.4759},
    3: {"Precision": 0.5422, "Recall": 0.6585, "DiceScore": 0.5917, "Accuracy": 0.8363, "CLDiceScore": 0.4686},
    4: {"Precision": 0.5429, "Recall": 0.4202, "DiceScore": 0.4642, "Accuracy": 0.8377, "CLDiceScore": 0.3487},
}

approach9= {
    0: {"Precision": 0.5648, "Recall": 0.6010, "DiceScore": 0.5806, "Accuracy": 0.8374, "CLDiceScore": 0.4786},
    1: {"Precision": 0.6256, "Recall": 0.6391, "DiceScore": 0.6308, "Accuracy": 0.8244, "CLDiceScore": 0.4719},
    2: {"Precision": 0.5543, "Recall": 0.6682, "DiceScore": 0.5980, "Accuracy": 0.8246, "CLDiceScore": 0.4805},
    3: {"Precision": 0.5913, "Recall": 0.6008, "DiceScore": 0.5859, "Accuracy": 0.8503, "CLDiceScore": 0.5102},
    4: {"Precision": 0.5845, "Recall": 0.6272, "DiceScore": 0.6040, "Accuracy": 0.8437, "CLDiceScore": 0.4825},
}

baseline_grayscale = {
    0: {"Precision": 0.6847, "Recall": 0.3061, "DiceScore": 0.3642, "Accuracy": 0.8222, "CLDiceScore": 0.3211},
    1: {"Precision": 0.5940, "Recall": 0.3265, "DiceScore": 0.3955, "Accuracy": 0.7941, "CLDiceScore": 0.3715},
    2: {"Precision": 0.6585, "Recall": 0.2905, "DiceScore": 0.3422, "Accuracy": 0.8192, "CLDiceScore": 0.3084},
    3: {"Precision": 0.6354, "Recall": 0.2208, "DiceScore": 0.2730, "Accuracy": 0.8240, "CLDiceScore": 0.2654},
    4: {"Precision": 0.6506, "Recall": 0.1889, "DiceScore": 0.2589, "Accuracy": 0.8196, "CLDiceScore": 0.2459},
}

baseline_3ChannelWindowing = {
    0: {"Precision": 0.4339, "Recall": 0.5488, "DiceScore": 0.4825, "Accuracy": 0.7693, "CLDiceScore": 0.4957},
    1: {"Precision": 0.4479, "Recall": 0.5558, "DiceScore": 0.4948, "Accuracy": 0.7320, "CLDiceScore": 0.5193},
    2: {"Precision": 0.3999, "Recall": 0.5096, "DiceScore": 0.4459, "Accuracy": 0.7490, "CLDiceScore": 0.4655},
    3: {"Precision": 0.3523, "Recall": 0.4823, "DiceScore": 0.4065, "Accuracy": 0.7404, "CLDiceScore": 0.4379},
    4: {"Precision": 0.3720, "Recall": 0.4754, "DiceScore": 0.4152, "Accuracy": 0.7452, "CLDiceScore": 0.4331},
}

approaches = {
    "Approach 1: FADA-Grayscale-Windowing-500to600": approach1,
    "Approach 2: FADA-Grayscale-Windowing-NoPretraining": approach2,
    "Approach 3: FADA-3ChannelWindowing": approach3,
    "Approach 4: FADA-Grayscale-Windowing-400to500": approach4,
    "Approach 5: FADA-Grayscale-Windowing-600to800": approach5,
    "Approach 6: FADA-Ensemble": approach6,
    "Approach 7: FADA-Grayscale-Windowing-500to600-CLDice": approach7,
    "Approach 8: FADA-Grayscale-CycleGAN-CNN": approach8,
    "Approach 9: FADA-Grayscale-CycleGAN-1x1Conv": approach9,
    "Baseline Grayscale": baseline_grayscale,
    "Baseline 3-Channel Windowing": baseline_3ChannelWindowing,
}

metrics = ["Precision", "Recall", "DiceScore", "Accuracy", "CLDiceScore"]
images = range(5)

# Plot settings
fig, axes = plt.subplots(3, 2, figsize=(20, 15))
axes = axes.flatten()
handles, labels = [], []

gap = 0.3        # Extra gap between groups
width = 0.1      # Bar width
num_approaches = len(approaches)
group_width = width * num_approaches + gap  # Total width for each group of bars

for i, metric in enumerate(metrics):
    ax = axes[i]
    # Multiply the image index by group_width to add space between groups
    x = np.arange(len(images)) * group_width

    for j, (approach_name, approach_data) in enumerate(approaches.items()):
        values = [approach_data[img][metric] for img in images]
        bars = ax.bar(x + j * width, values, width, label=approach_name)

        if i == 0:  # Only add labels once for the legend
            handles.append(bars[0])
            labels.append(approach_name)

    ax.set_title(f"{metric} Comparison", fontsize=16)
    # Center the xticks within each group
    ax.set_xticks(x + width * (num_approaches - 1) / 2)
    ax.set_xticklabels([f"Test {img}" for img in images], fontsize=12)
    ax.set_xlabel("Testset", fontsize=14)
    ax.set_ylabel(metric, fontsize=14)
    ax.grid(axis="y", linestyle="--", alpha=0.7)

# Hide any unused subplot (the 6th axis in this case)
for i in range(len(metrics), len(axes)):
    axes[i].axis("off")

fig.legend(
    handles,
    labels,
    loc="upper center",
    ncol=num_approaches,
    fontsize=12,
    bbox_to_anchor=(0.5, 1.03),
)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

In [ ]:
average_metrics = {approach_name: {metric: np.mean([data[image][metric] for image in images])
                                   for metric in metrics}
                   for approach_name, data in approaches.items()}

gap = 0.3        # Extra gap between metric groups
width = 0.1      # Bar width
num_approaches = len(approaches)
group_width = width * num_approaches + gap  # Total width for each metric group

# Scale the x-axis positions by the group width to add extra space between groups
x = np.arange(len(metrics)) * group_width  

fig, ax = plt.subplots(figsize=(16, 12))

for i, (approach_name, metrics_data) in enumerate(average_metrics.items()):
    values = [metrics_data[metric] for metric in metrics]
    ax.bar(x + i * width, values, width, label=approach_name)

ax.set_title("Average Metric Comparison Across Approaches", fontsize=14)
# Center the xticks within each metric group
ax.set_xticks(x + width * (num_approaches - 1) / 2)
ax.set_xticklabels(metrics, fontsize=12)
ax.set_xlabel("Metrics", fontsize=12)
ax.set_ylabel("Average Value", fontsize=12)
ax.legend(fontsize=10)
ax.grid(axis="y", linestyle="--", alpha=0.7)

plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
from IPython.display import display

# # Data
# approach1 = {
#     0: {"Precision": 0.5352, "Recall": 0.6670, "DiceScore": 0.5910, "Accuracy": 0.8196},
#     1: {"Precision": 0.5303, "Recall": 0.7830, "DiceScore": 0.6316, "Accuracy": 0.7854},
#     2: {"Precision": 0.5292, "Recall": 0.6412, "DiceScore": 0.5739, "Accuracy": 0.8251},
#     3: {"Precision": 0.5170, "Recall": 0.6194, "DiceScore": 0.5522, "Accuracy": 0.8223},
#     4: {"Precision": 0.5572, "Recall": 0.6122, "DiceScore": 0.5679, "Accuracy": 0.8267},
# }

# approach2 = {
#     0: {"Precision": 0.5215, "Recall": 0.3836, "DiceScore": 0.4391, "Accuracy": 0.8105},
#     1: {"Precision": 0.5370, "Recall": 0.6201, "DiceScore": 0.5739, "Accuracy": 0.7871},
#     2: {"Precision": 0.5145, "Recall": 0.4671, "DiceScore": 0.4838, "Accuracy": 0.8076},
#     3: {"Precision": 0.4186, "Recall": 0.6554, "DiceScore": 0.5100, "Accuracy": 0.7725},
#     4: {"Precision": 0.4513, "Recall": 0.5585, "DiceScore": 0.4962, "Accuracy": 0.7865},
# }

# approach3 = {
#     0: {"Precision": 0.5057, "Recall": 0.7008, "DiceScore": 0.5851, "Accuracy": 0.8076},
#     1: {"Precision": 0.5496, "Recall": 0.6445, "DiceScore": 0.5901, "Accuracy": 0.7969},
#     2: {"Precision": 0.4882, "Recall": 0.6959, "DiceScore": 0.5640, "Accuracy": 0.7975},
#     3: {"Precision": 0.4675, "Recall": 0.5350, "DiceScore": 0.4882, "Accuracy": 0.8040},
#     4: {"Precision": 0.4793, "Recall": 0.5797, "DiceScore": 0.5174, "Accuracy": 0.8007},
# }

# baseline_grayscale = {
#     0: {"Precision": 0.6847, "Recall": 0.3061, "DiceScore": 0.3642, "Accuracy": 0.8222},
#     1: {"Precision": 0.5940, "Recall": 0.3265, "DiceScore": 0.3955, "Accuracy": 0.7941},
#     2: {"Precision": 0.6585, "Recall": 0.2905, "DiceScore": 0.3422, "Accuracy": 0.8192},
#     3: {"Precision": 0.6354, "Recall": 0.2208, "DiceScore": 0.2730, "Accuracy": 0.8240},
#     4: {"Precision": 0.6506, "Recall": 0.1889, "DiceScore": 0.2589, "Accuracy": 0.8196},
# }

# baseline_3ChannelWindowing = {
#     0: {"Precision": 0.4339, "Recall": 0.5488, "DiceScore": 0.4825, "Accuracy": 0.7693},
#     1: {"Precision": 0.4479, "Recall": 0.5558, "DiceScore": 0.4948, "Accuracy": 0.7320},
#     2: {"Precision": 0.3999, "Recall": 0.5096, "DiceScore": 0.4459, "Accuracy": 0.7490},
#     3: {"Precision": 0.3523, "Recall": 0.4823, "DiceScore": 0.4065, "Accuracy": 0.7404},
#     4: {"Precision": 0.3720, "Recall": 0.4754, "DiceScore": 0.4152, "Accuracy": 0.7452},
# }

# approaches = {
#     "Approach 1: FADA-Grayscale-Windowing:": approach1,
#     "Approach 2: FADA-Grayscale-Windowing-NoPretraining": approach2,
#     "Approach 3: FADA-3ChannelWindowing": approach3,
#     "Baseline Grayscale": baseline_grayscale,
#     "Baseline 3-Channel Windowing": baseline_3ChannelWindowing,
# }

metrics = ["Precision", "Recall", "DiceScore", "Accuracy", "CLDiceScore"]
images = range(5)

# Prepare data for the DiceScore table with approaches as rows
dice_scores = []

# Extract DiceScores for each approach and image
for approach_name, data in approaches.items():
    row = {"Approach": approach_name}
    for image in images:
        test_indices = set(images) - {image}
        row[f"{test_indices}"] = round(data[image]["CLDiceScore"], 4)
    # Add the average DiceScore for the approach
    row["Averaged"] = round(np.mean([data[image]["CLDiceScore"] for image in images]), 4)
    dice_scores.append(row)

# Convert to DataFrame
dice_df_transposed = pd.DataFrame(dice_scores)

dice_df_transposed.head(n=11)